## 1. Install and import

In [ ]:
!pip uninstall -y -q torchao
!pip install -q -U transformers peft trl datasets accelerate rouge_score bert_score sentencepiece

import os, re, json, gc, time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset, Dataset
from peft import LoraConfig, PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM
from trl import SFTConfig, SFTTrainer

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
os.makedirs("results", exist_ok=True)

## 2. Load data

`ccdv/arxiv-summarization` (paper body > abstract). Only a subset is used: **4,000 training papers** and **200 test papers**.

In [ ]:
N_TRAIN, N_TEST = 4000, 200

def load_split(split, n):
    # Falls back to the parquet branch if dataset scripts are not supported by the installed version
    last_err = None
    for kw in [dict(), dict(revision="refs/convert/parquet")]:
        try:
            ds = load_dataset("ccdv/arxiv-summarization", split=split, streaming=True, **kw)
            if split == "train":
                ds = ds.shuffle(seed=42, buffer_size=2000)
            return list(ds.take(n))
        except Exception as e:
            last_err = e
    raise last_err

train_raw = load_split("train", N_TRAIN)
test_raw = load_split("test", N_TEST)
print(len(train_raw), "train |", len(test_raw), "test")

## 3. Preprocess

Papers have a median of about 4,850 words, so only the **first 1,500 tokens** of each paper are used as input
(abstracts are capped at 400 tokens). The same truncation is used for training and testing.

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_ARTICLE_TOKENS, MAX_ABSTRACT_TOKENS = 1500, 400
INSTRUCTION = "Summarize the following research paper as a concise abstract."

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def clean(text):
    text = re.sub(r"</?S>", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def truncate_tokens(text, max_tokens):
    ids = tokenizer(text, add_special_tokens=False).input_ids[:max_tokens]
    return tokenizer.decode(ids)

def build(ex):
    article = truncate_tokens(clean(ex["article"])[:15000], MAX_ARTICLE_TOKENS)
    abstract = truncate_tokens(clean(ex["abstract"]), MAX_ABSTRACT_TOKENS)
    return {"prompt": [{"role": "user", "content": f"{INSTRUCTION}\n\n{article}"}],
            "completion": [{"role": "assistant", "content": abstract}]}

train_formatted = [build(e) for e in train_raw]
test_formatted = [build(e) for e in test_raw]
test_refs = [clean(e["abstract"]) for e in test_raw]
test_articles = [clean(e["article"])[:15000] for e in test_raw]   # used by the baseline models

lengths = [len(tokenizer(tokenizer.apply_chat_template(e["prompt"] + e["completion"], tokenize=False)).input_ids)
           for e in train_formatted[:500]]
print(f"Tokens per training example (first 500): median {int(np.median(lengths))}, max {max(lengths)}")

## 4. Fine-tune with LoRA

LoRA (rank 16) on all attention and MLP projections, 1 epoch, effective batch size 16, fp16 (the T4 does not support bf16).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUTPUT_DIR = "/content/drive/MyDrive/qwen05b-arxiv-lora"

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32, attn_implementation="sdpa")
model.config.use_cache = False

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=8,
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_length=2048,
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    report_to="none",
)

trainer = SFTTrainer(model=model, args=args,
                     train_dataset=Dataset.from_list(train_formatted),
                     processing_class=tokenizer, peft_config=lora_config)
trainer.model.print_trainable_parameters()
trainer.train()

trainer.save_model(OUTPUT_DIR + "/final")
tokenizer.save_pretrained(OUTPUT_DIR + "/final")

del trainer, model
gc.collect(); torch.cuda.empty_cache()

## 5. Generate summaries on the 200 test papers

Fine-tuned Qwen and zero-shot Qwen (no adapter) use identical decoding settings: greedy decoding, up to 300 new tokens,
`no_repeat_ngram_size=12`.

In [ ]:
tokenizer.padding_side = "left"
GEN_KWARGS = dict(max_new_tokens=300, do_sample=False, no_repeat_ngram_size=12,
                  pad_token_id=tokenizer.pad_token_id)

def generate_all(model, examples, batch_size=8):
    prompts = [tokenizer.apply_chat_template(e["prompt"], tokenize=False, add_generation_prompt=True)
               for e in examples]
    outputs = []
    for i in range(0, len(prompts), batch_size):
        enc = tokenizer(prompts[i:i + batch_size], return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            out = model.generate(**enc, **GEN_KWARGS)
        outputs += [t.strip() for t in tokenizer.batch_decode(out[:, enc["input_ids"].shape[1]:],
                                                              skip_special_tokens=True)]
    return outputs

def save_outputs(name, outs):
    json.dump(outs, open(f"results/outputs_{name}.json", "w"))

# Fine-tuned model
base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float16).to("cuda")
ft_model = PeftModel.from_pretrained(base, OUTPUT_DIR + "/final").merge_and_unload().eval()
save_outputs("finetuned", generate_all(ft_model, test_formatted))
del ft_model, base
gc.collect(); torch.cuda.empty_cache()

# Zero-shot control (same base model, no fine-tuning)
zs_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float16).to("cuda").eval()
save_outputs("zeroshot", generate_all(zs_model, test_formatted))
del zs_model
gc.collect(); torch.cuda.empty_cache()
json.dump(test_refs, open("results/references.json", "w"))
print("Qwen outputs saved")

## 6. Baselines 
(`summarize: ` prefix, 1024-token input, 6 beams, 100-150 output tokens), on the same 200 papers.

In [ ]:
BASELINES = {"bart": "facebook/bart-large-cnn",
              "t5": "google-t5/t5-base",
              "pegasus": "google/pegasus-cnn_dailymail"}

def run_baselines(model_id, batch_size=4):
    tok = AutoTokenizer.from_pretrained(model_id)
    mdl = AutoModelForSeq2SeqLM.from_pretrained(model_id, dtype=torch.float32).to("cuda").eval()
    outs = []
    for i in range(0, len(test_articles), batch_size):
        batch = ["summarize: " + a for a in test_articles[i:i + batch_size]]
        enc = tok(batch, max_length=1024, truncation=True, padding=True, return_tensors="pt").to("cuda")
        with torch.no_grad():
            ids = mdl.generate(**enc, max_length=150, min_length=100, length_penalty=1.0,
                               num_beams=6, early_stopping=False)
        outs += [s.replace("<n>", " ").strip() for s in tok.batch_decode(ids, skip_special_tokens=True)]
    del mdl
    gc.collect(); torch.cuda.empty_cache()
    return outs

for name, model_id in BASELINES.items():
    save_outputs(name, run_baselines(model_id))
    print(name, "done")

## 7. Save outputs

In [ ]:
# Keep a copy on Google Drive in case the Colab session ends
import shutil
shutil.copytree("results", "/content/drive/MyDrive/qwen-arxiv-results", dirs_exist_ok=True)
print(sorted(os.listdir("results")))